In [1]:
import numpy as np


In [3]:
with open("corpus.txt", "r") as f:
    words = [line.strip() for line in f if line.strip()]

print("Corpus:")
print(words)

Corpus:
['low', 'lower', 'lowest', 'new', 'newer', 'widest']


In [4]:
text = " ".join(words)

tokens = list(text)

vocab = sorted(set(tokens))

token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

token_ids = np.array([token_to_id[t] for t in tokens])

print("\nVocabulary:")
print(vocab)

print("\nToken IDs:")
print(token_ids)


Vocabulary:
[' ', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Token IDs:
[ 4  6 10  0  4  6 10  2  7  0  4  6 10  2  8  9  0  5  2 10  0  5  2 10
  2  7  0 10  3  1  2  8  9]


In [5]:
inputs = token_ids[:-1]
targets = token_ids[1:]

print("\nInput Shape:", inputs.shape)
print("Target Shape:", targets.shape)


Input Shape: (32,)
Target Shape: (32,)


In [6]:
vocab_size = len(vocab)
embedding_dim = 8
np.random.seed(42)
embedding_matrix = np.random.randn(vocab_size, embedding_dim)
embedded = embedding_matrix[inputs]
print("\nEmbedding Shape:", embedded.shape)


Embedding Shape: (32, 8)


In [7]:
seq_len = embedded.shape[0]
position = np.arange(seq_len).reshape(-1,1)
dimension = np.arange(embedding_dim).reshape(1,-1)
angle_rates = 1 / np.power(10000, (2*(dimension//2))/embedding_dim)
angles = position * angle_rates
pos_encoding = np.zeros((seq_len, embedding_dim))
pos_encoding[:,0::2] = np.sin(angles[:,0::2])
pos_encoding[:,1::2] = np.cos(angles[:,1::2])
embedded = embedded + pos_encoding
print("\nPositional Encoding Added")


Positional Encoding Added


In [10]:
mask = np.triu(np.ones((seq_len, seq_len)), k=1)
mask = mask * (-1e9)

WQ = np.random.randn(embedding_dim, embedding_dim)
WK = np.random.randn(embedding_dim, embedding_dim)
WV = np.random.randn(embedding_dim, embedding_dim)

Q = embedded @ WQ
K = embedded @ WK
V = embedded @ WV

scores = (Q @ K.T) / np.sqrt(embedding_dim)
scores = scores + mask
exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attention = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
context = attention @ V
print("\nAttention Shape:", context.shape)


Attention Shape: (32, 8)


In [12]:
W_out = np.random.randn(embedding_dim, vocab_size)
logits = context @ W_out

exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
print("\nSoftmax Output Shape:", probabilities.shape)


Softmax Output Shape: (32, 11)


In [13]:
loss = -np.mean(np.log(probabilities[np.arange(len(targets)), targets] + 1e-9))
print("\nCross Entropy Loss:", loss)



Cross Entropy Loss: 12.095370299426754


In [14]:
grad = probabilities.copy()
grad[np.arange(len(targets)), targets] -= 1
grad /= len(targets)
learning_rate = 0.01
dW_out = context.T @ grad
W_out -= learning_rate * dW_out
print("\nBackpropagation Completed")


Backpropagation Completed


In [16]:
current = inputs[-1]
generated = []
for i in range(10):
    x = embedding_matrix[current]
    logits = x @ W_out
    probs = np.exp(logits - np.max(logits))
    probs /= np.sum(probs)
    current = np.argmax(probs)
    generated.append(id_to_token[current])

print("\nGenerated Next Tokens:")
print("".join(generated))


Generated Next Tokens:
ssssssssss
